Iteration of the trade matrix 

In [ ]:
import pandas as pd
import numpy as np
import os

# The trade matrix for each product needs to be iterated; here we take one as an example.
# Read Excel data
# A sheet: 32 rows and 34 columns (including interprovincial transfers, exports, export ratio) Production statistics
A_path = r'E:\Trade matrix.xlsx'  
A = pd.read_excel(A_path, header=None)

# Consumption data: Province + Values
consumption_path = r'E:\Consumption.xlsx'  
consumption_df = pd.read_excel(consumption_path)
consumption_df.columns = ['province', 'consumption']
consumption_df.set_index('province', inplace=True)  # Correctly use inplace

# Actual production data: Province + Values
actual_production_path = r'E:\Statistics_production.xlsx'  
production_df = pd.read_excel(actual_production_path)
production_df.columns = ['province', 'production']
production_df.set_index('province', inplace=True)   # Correctly use inplace

# 2. Align consumption and production order
def clean_name(name):
    return name.strip().replace("Province", "").replace("City", "").replace("Autonomous region", "").replace("Zhuang ethnic group", "").replace("Hui people", "").replace("Uighur", "")

# Clean all province names
provinces_raw = A.iloc[1:, 0].values
provinces_cleaned = pd.Series(provinces_raw).map(clean_name).values

consumption_df.index = consumption_df.index.map(clean_name)
production_df.index = production_df.index.map(clean_name)

# Print verification
print("Cleaned provinces in A:", provinces_cleaned.tolist())
print("Consumption data index:", list(consumption_df.index))
print("Production data index:", list(production_df.index))

# Get data aligned with A's order
consumption = consumption_df.loc[provinces_cleaned, 'consumption']
actual_production = production_df.loc[provinces_cleaned, 'production']
# -------------------------------
# 3. Define iteration function
def iterate_trade_matrix(A, consumption, actual_production, tolerance=0.05
                         , max_iter=20000, verbose=False):
    provinces = A.iloc[1:, 0].values
    n = len(provinces)

    transfer_matrix = A.iloc[1:n+1, 1:n+1].astype(float).values

    export_vector = A.iloc[1:n+1, 32].astype(float).values
    export_ratio = A.iloc[1:n+1, 33].astype(float).values
    for iteration in range(max_iter):
        column_sums = transfer_matrix.sum(axis=0)    # Sum of each column, gives total input for each consuming province
        column_sums[column_sums == 0] = 1  # Avoid division by 0
        proportion_matrix = transfer_matrix / column_sums  # Proportion of consumption coming from other provinces
        consumption_matrix = proportion_matrix * consumption.values.reshape(1, -1)  # Distribute consumption data according to proportion, resulting in a new interprovincial "consumption source matrix"
        estimated_production_noex = consumption_matrix.sum(axis=1) 
        ex = estimated_production_noex * export_ratio / (1 - export_ratio)
        estimated_production = estimated_production_noex + ex
         # Derived production for each province = consumption sent to other provinces + export volume
        diff_ratio = 1 - (actual_production.values / estimated_production)  # Calculate the difference ratio between derived and actual production
        print(diff_ratio)
        if verbose:
            max_error = np.max(np.abs(diff_ratio))
            print(f"Iteration {iteration+1}: Maximum error = {max_error:.4f}")  # If verbose=True, output the max error of each iteration for monitoring convergence

        if np.all(np.abs(diff_ratio) < tolerance):
            break  # Stop iteration when the error for all provinces is less than the tolerance

        for i in range(n):
            transfer_matrix[i, :] -= transfer_matrix[i, :] * diff_ratio[i]
            export_vector[i] -= export_vector[i] * diff_ratio[i]     # Scale the trade flows and export values of each province by the error ratio
    # Final trade matrix and recalculated export
    column_sums = transfer_matrix.sum(axis=0)
    column_sums[column_sums == 0] = 1
    proportion_matrix = transfer_matrix / column_sums
    final_consumption_matrix = proportion_matrix * consumption.values.reshape(1, -1)  # Re-generate the consumption distribution matrix (for output)

    new_export_vector = export_ratio * transfer_matrix.sum(axis=1) / (1 - export_ratio)  # Recalculate export volume using export ratio and total transfer to ensure consistency

    final_trade_matrix = pd.DataFrame(transfer_matrix, index=provinces, columns=provinces)
    final_export_series = pd.Series(new_export_vector, index=provinces, name='Export')
    final_consumption_matrix_df = pd.DataFrame(final_consumption_matrix, index=provinces, columns=provinces)  # Convert NumPy array into Pandas object for output and visualization

    # Derived production = total interprovincial transfer + export
    derived_output = transfer_matrix.sum(axis=1) + new_export_vector
    output_diff_ratio = 1 - (actual_production.values / derived_output)

    output_comparison = pd.DataFrame({
        'Province': provinces,
        'Derived Production': derived_output,
        'Actual Production': actual_production.values,
        'Difference Ratio': output_diff_ratio
    })

    return final_trade_matrix, final_export_series, final_consumption_matrix_df, output_comparison

# 4. Run the model
final_trade_matrix, final_export, final_consumption_matrix, output_comparison = iterate_trade_matrix(
    A, consumption, actual_production, tolerance=0.05, max_iter=20000, verbose=True)

# 5. Save all results
output_path = r'E:\China Nitrogen Cycle Data\Iterative Data'
os.makedirs(output_path, exist_ok=True)  
final_trade_matrix.to_excel(os.path.join(output_path, 'Trade matrix_after Iteration.xlsx'))
final_export.to_excel(os.path.join(output_path, 'Export_after Iteration.xlsx'))


